<style>
  .cell-markdown { overflow: auto !important; }
  .mermaid { max-width: 100%; height: auto; }
</style>

# Cumulative Windows Walkthrough


## Overview

[Preparation](#prep)

* [Topology](#topology)
* [Steps](#steps)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [2]:
import sys
sys.path.insert(1, "../..")
sys.path.insert(1, "../../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"

#

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)


<a id="topology"></a>
---
## Topology

Now it's time for the walkthrough itself. We go for a slightly simpler example for the walkthroughs.

The corresponding test can be found here: [test_windows.py](../../../../test/streams/test_windows.py)

In [13]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = 100
step_int = size_int // 5
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    # 1. Select customer_id, price and ts from the value.
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    # 2. Expire with window size order_generator.ts_step_int * 100, advance size 100 / 5 = 20 and allowed_lateness = window_size * 2,  
    .expire_cumulative(lambda r: r["ts"], size_int, step_int, allowed_lateness_int)
    # 3. Deduplicate.
    .distinct()
)
#
# 4. Set up the window: group by customer ID, count the orders, sum up the prices of the orders and get the last timestamp of the window.
sink_tn = order_tn.group_by_agg_cumulative(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    step_int=step_int,
    key_fun=lambda r: r["customer_id"],
            agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                      "total_price": agg_r["total_price"] + r["price"],
                                      "last_ts": max(agg_r["last_ts"], r["ts"])},
            agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0},
            project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                                "orders": agg_r["orders"],
                                                "total_price": agg_r["total_price"],
                                                "last_ts": agg_r["last_ts"]},
    trigger_positive_only_bool=False
).sink(sink_str)
#
_ = built_tn = Tn.build(sink_tn)

What do we do?
1. `map()`: Select customer_id, price and ts from the value.
2. `expire_cumulative()`: Expire with window size `100`, advance size `100 // 5 = 20` and `allowed_lateness` = `size_int * 2 = 200`.  
3. `distinct()`: Deduplicate.
4. `group_by_agg_cumulative()`: Set up the window: group by customer ID, count the orders sum up the prices of the orders and get the last timestamp of the window.

Next, we illustrate how the tumbling window works by processing some example data - one by one, in baby steps.

We use two types of illustrations in each step:
1. Top-down view:
  * time proceeds from top to bottom (starting with `0`)
  * small grey circles mark the time every `100` ms for clarity
  * the latest timestamp of the input after the respective step is written at the top 
  * new events coming in a step are indicated a blue frame
  * old events have a grey frame
  * the triggered outputs in the sink are indicated by green color
2. Left-right view:
  * time proceeds from left to right (starting with `0`)
  * new windows appear below the old windows
  * `^`: latest timestamp of this step
  * `(^)`: latest timestamp of the previous step
  * `<<<`: time window(s) containing the event from this step
  * `!!!`: time window(s) triggered by the event from this step



<a id="steps"></a>
## Steps

### Step 1

In step 1, the first order from customer 1 arrives at timestamp 10:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 10"]
        direction TB
        0(("0")) e1@-.-> 10
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#1565c0,stroke-width:6px
```

In step 1, first, the first order arrives from customer 1 at timestamp 10:

This event:
* advances the latest timestamp from `0` (start, visualized by `(^)`) to `10` (visualized by `^`),
* and falls into the first cumulative windows `[0, 20)`, `[0, 40)`, `[0, 60)`, `[0, 80)`, `[0, 100)` (visualized by `<<<`):
```
[0 -- 20) <<<
[0 ---- 40) <<<
[0 ------ 60) <<<
[0 -------- 80) <<<
[0 ---------- 100) <<<
(^) 
   ^
```

As the latest timestamp is not yet beyond the end of the first cumulative window, no output is triggered. Why?

Because the default `trigger_fun` is defined as `lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1]` where `r_end_ts_tuple[1] = end_ts = 20` (`20` being the end of the first of the windows). At this point, `latest_ts = 10 >= 20` evaluates to `False` and consequently, no output is triggered.

Let's see this happening for real:

In [14]:
process(built_tn, customer_id=1, price=100, ts=10, w=1)


[20, 40, 60, 80, 100]
[20, 40, 60, 80, 100]
Triggers:


### Step 2

In step 2, the second order arrives from customer 1 at timestamp 30:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 30"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 30
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 100,\n&quot;last_ts&quot;: 10\n&quot;window_end&quot;: 20}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    30 --- space1
    linkStyle 2 stroke:none
    30 Link@== Triggers ==> Output
    linkStyle 3 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `10` to `30`,
* falls into the windows `[0, 40)`, `[0, 60)`, `[0, 80)`, `[0, 100)`
* and triggers the first window `[0, 20)`
``` 
[0 -- 20) !!!
[0 ---- 40) <<<
[0 ------ 60) <<<
[0 -------- 80) <<<
[0 ---------- 100) <<<
(^) 
     ^
```


In [15]:
process(built_tn, customer_id=1, price=200, ts=30, w=1)


[40, 60, 80, 100]
[40, 60, 80, 100]
Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 100, 'last_ts': 10, 'window_end': 20}


### Step 3

In step 3, an order from customer 2 arrives at timestamp `75`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 75"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 30 e3@-.-> 75
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 100,\n&quot;last_ts&quot;: 10\n&quot;window_end&quot;: 20}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    75 --- space1
    linkStyle 3 stroke:none
    75 Link@== Triggers ==> Output
    linkStyle 4 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `30` to `75`,
* falls into the windows `[0, 20)`, `[0, 40)`, `[0, 60)`, `[0, 80)`, `[0, 100)`
* and triggers windows `[0, 40)` and `[0, 60)`
``` 
[0 -- 20)
[0 ---- 40) !!!
[0 ------ 60) !!!
[0 -------- 80) <<<
[0 ---------- 100) <<<
(^) 
           ^
```


In [16]:
process(built_tn, customer_id=2, price=50, ts=75, w=1)

[80, 100]
[80, 100]
Triggers:
{'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 30, 'window_end': 40}
{'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 30, 'window_end': 60}


### Step 4

An order from customer 2 arrives out-of-order at timestamp `15`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 75"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 15 e3@-.-> 30 e4@-.-> 75
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 150,\n&quot;last_ts&quot;: 15\n&quot;window_end&quot;: 20}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    15 Link@== Triggers ==> Output1
    linkStyle 4 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 3,\n&quot;total_price&quot;: 350,\n&quot;last_ts&quot;: 30\n&quot;window_end&quot;: 40}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    15 Link@== Triggers ==> Output2
    linkStyle 5 stroke:#00bb00,stroke-width:3px

    Output3@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 3,\n&quot;total_price&quot;: 350,\n&quot;last_ts&quot;: 30\n&quot;window_end&quot;: 60}"}
    style Output3 fill:none,stroke:#00bb00,stroke-width:6px
    15 Link@== Triggers ==> Output3
    linkStyle 6 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `30` to `75`,
* falls into the windows `[0, 20)`, `[0, 40)`, `[0, 60)`, `[0, 80)` and `[0, 100)`
* and triggers the corrections of the windows `[0, 20)`, `[0, 40)` and `[0, 60)`
``` 
[0 -- 20) <<< !!!
[0 ---- 40) <<< !!!
[0 ------ 60) <<< !!!
[0 -------- 80) <<<
[0 ---------- 100) <<<
          (^) 
           ^
```


In [17]:
process(built_tn, customer_id=1, price=50, ts=15, w=1)


[20, 40, 60, 80, 100]
[20, 40, 60, 80, 100]
Triggers:
{'customer_id': 1, 'orders': 2, 'total_price': 150, 'last_ts': 15, 'window_end': 20}
{'customer_id': 1, 'orders': 3, 'total_price': 350, 'last_ts': 30, 'window_end': 40}
{'customer_id': 1, 'orders': 3, 'total_price': 350, 'last_ts': 30, 'window_end': 60}


### Step 5

An order from customer 1 arrives at timestamp `105`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 15 e3@-.-> 30 e4@-.-> 75 e5@-.-> 100(("100")) e6@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 3,\n&quot;total_price&quot;: 350,\n&quot;last_ts&quot;: 30\n&quot;window_end&quot;: 80}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    105 Link@== Triggers ==> Output1
    linkStyle 6 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 3,\n&quot;total_price&quot;: 350,\n&quot;last_ts&quot;: 30\n&quot;window_end&quot;: 100}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    105 Link@== Triggers ==> Output2
    linkStyle 7 stroke:#00bb00,stroke-width:3px

    Output3@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50,\n&quot;last_ts&quot;: 75\n&quot;window_end&quot;: 80}"}
    style Output3 fill:none,stroke:#00bb00,stroke-width:6px
    105 Link@== Triggers ==> Output3
    linkStyle 8 stroke:#00bb00,stroke-width:3px

    Output4@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50,\n&quot;last_ts&quot;: 75\n&quot;window_end&quot;: 100}"}
    style Output4 fill:none,stroke:#00bb00,stroke-width:6px
    105 Link@== Triggers ==> Output4
    linkStyle 9 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `70` to `105`,
* falls into the windows `[0, 120)`, `[0, 140)`, `[0, 160)`, `[0, 180)` and `[0, 200)`
* and triggers the windows `[0, 80)` and `[0, 100)`:
``` 
[0 -- 20)
[0 ---- 40)
[0 ------ 60)
[0 -------- 80) !!!
[0 ---------- 100) !!!
[0 ------------ 120) <<<
[0 -------------- 140) <<<
[0 ---------------- 160) <<<
[0 ------------------ 180) <<<
[0 -------------------- 200) <<<
         (^) 
             ^
```


In [ ]:
process(built_tn, customer_id=3, price=400, ts=105, w=1)

[120, 140, 160, 180, 200]
[120, 140, 160, 180, 200]
Triggers:
{'customer_id': 1, 'orders': 3, 'total_price': 350, 'last_ts': 30, 'window_end': 80}
{'customer_id': 1, 'orders': 3, 'total_price': 350, 'last_ts': 30, 'window_end': 100}
{'customer_id': 2, 'orders': 1, 'total_price': 50, 'last_ts': 75, 'window_end': 80}
{'customer_id': 2, 'orders': 1, 'total_price': 50, 'last_ts': 75, 'window_end': 100}


### Step 6

An order from customer 1 arrives at timestamp `410`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 410"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 15 e3@-.-> 30 e4@-.-> 75 e5@-.-> 100(("100")) e6@-.-> 105 e7@-.-> 200(("200")) e8@-.-> 300(("300")) e9@-.-> 400(("400")) e10@-.-> 410
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    style 400 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    410@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 10,\n&quot;ts&quot;: 410}"}
    style 410 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 500,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 120}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    410 Link@== Triggers ==> Output1
    linkStyle 10 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 500,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 140}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    410 Link@== Triggers ==> Output2
    linkStyle 11 stroke:#00bb00,stroke-width:3px

    Output3@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 500,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 160}"}
    style Output3 fill:none,stroke:#00bb00,stroke-width:6px
    410 Link@== Triggers ==> Output3
    linkStyle 12 stroke:#00bb00,stroke-width:3px

    Output4@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 500,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 180}"}
    style Output4 fill:none,stroke:#00bb00,stroke-width:6px
    410 Link@== Triggers ==> Output4
    linkStyle 13 stroke:#00bb00,stroke-width:3px

    Output5@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 500,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 200}"}
    style Output5 fill:none,stroke:#00bb00,stroke-width:6px
    410 Link@== Triggers ==> Output5
    linkStyle 14 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `105` to `410`,
* falls into the windows `[0, 420)`, `[0, 440)`, `[0, 460)`, `[0, 480)` and `[0, 500)`
* and triggers the windows `[0, 120)`,  `[0, 140)`, `[0, 160)`, `[0, 180)` and `[0, 200)`:
``` 
[0 -- 20)
[0 ---- 40)
[0 ------ 60)
[0 -------- 80)
[0 ---------- 100)
[0 ------------ 120) !!!
[0 -------------- 140) !!!
[0 ---------------- 160) !!!
[0 ------------------ 180) !!!
[0 -------------------- 200) !!!
[0 ---------------------- 220)
[0 ------------------------ 240)
[0 -------------------------- 260)
[0 ---------------------------- 280)
[0 ------------------------------ 300)
[0 -------------------------------- 320)
[0 ---------------------------------- 340)
[0 ------------------------------------ 360)
[0 -------------------------------------- 380)
[0 ---------------------------------------- 400)
[0 ------------------------------------------ 420) <<<
[0 -------------------------------------------- 440) <<<
[0 ---------------------------------------------- 460) <<<
[0 ------------------------------------------------ 480) <<<
[0 -------------------------------------------------- 500) <<<
            (^) 
                                           ^

```


In [19]:
process(built_tn, customer_id=1, price=10, ts=410, w=1)


[420, 440, 460, 480, 500]
[420, 440, 460, 480, 500]
[20, 40, 60, 80, 100]
[80, 100]
[40, 60, 80, 100]
[20, 40, 60, 80, 100]
Triggers:
{'customer_id': 3, 'orders': 1, 'total_price': 400, 'last_ts': 101, 'window_end': 120}
{'customer_id': 3, 'orders': 1, 'total_price': 400, 'last_ts': 101, 'window_end': 140}
{'customer_id': 3, 'orders': 1, 'total_price': 400, 'last_ts': 101, 'window_end': 160}
{'customer_id': 3, 'orders': 1, 'total_price': 400, 'last_ts': 101, 'window_end': 180}
{'customer_id': 3, 'orders': 1, 'total_price': 400, 'last_ts': 101, 'window_end': 200}


### Step 7

An order from customer 1 arrives late at timestamp `5`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 410"]
        direction TB
        0(("0")) e1@-.-> 5 e2@-.-> 10 e3@-.-> 15 e4@-.-> 30 e5@-.-> 75 e6@-.-> 100(("100")) --> 105 e7@-.-> 200(("200")) e8@-.-> 300(("300")) e9@-.-> 400(("400")) e10@-.-> 410
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    style 400 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    410@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 10,\n&quot;ts&quot;: 410}"}
    style 410 fill:none,stroke:#333,stroke-width:6px

    5@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 999,\n&quot;ts&quot;: 5}"}
    style 5 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* does not advance the latest timestamp (it stays at `410`)
* falls into the windows `[0, 20)`, `[0, 40)`, `[0, 60)`, `[0, 80)` and `[0, 100)`
* and triggers nothing because it comes in too late: (`allowed_lateness = 200`): `410 (latest) - ( 100 (window end for the last window for ts = 5) + 100 (window buffer) = 200 ) = 210 > 200`, i.e., it is discarded right away:
``` 
[0 -- 20) !!!
[0 ---- 40) !!!
[0 ------ 60) !!!
[0 -------- 80) !!!
[0 ---------- 100) !!!
[0 ------------ 120)
[0 -------------- 140)
[0 ---------------- 160)
[0 ------------------ 180)
[0 -------------------- 200)
[0 ---------------------- 220)
[0 ------------------------ 240)
[0 -------------------------- 260)
[0 ---------------------------- 280)
[0 ------------------------------ 300)
[0 -------------------------------- 320)
[0 ---------------------------------- 340)
[0 ------------------------------------ 360)
[0 -------------------------------------- 380)
[0 ---------------------------------------- 400)
[0 ------------------------------------------ 420)
[0 -------------------------------------------- 440)
[0 ---------------------------------------------- 460)
[0 ------------------------------------------------ 480)
[0 -------------------------------------------------- 500)
                                          (^) 
                                           ^

```


In [20]:
process(built_tn, customer_id=1, price=999, ts=5, w=1)

[20, 40, 60, 80, 100]
Triggers:
